# Binary Response Models: Linear Probability, Logit, and Probit Frameworks

## 1. Introduction and Overview

In many enterprise applications, the outcome of interest is not a continuous magnitude but a discrete state: a customer churns or does not churn, a loan defaults or is repaid, a patient has a disease or is healthy.

Binary response models are a specialized class of Generalized Linear Models (GLMs). They model the conditional probability P(Y=1|X) of a dichotomous outcome. Unlike Ordinary Least Squares (OLS), which assumes a continuous, unbounded dependent variable with constant variance, binary models require non-linear link functions to bound predictions between 0 and 1 and account for the heteroskedasticity inherent in Bernoulli distributions.

This notebook provides a rigorous technical analysis of binary response models. We will explore the Linear Probability Model (LPM), Logit, and Probit models, establishing the mathematical foundations for interpreting coefficients through log-odds, odds ratios, and marginal effects.

In [ ]:
# Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
import warnings

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("colorblind")

# Set random seed for reproducibility
np.random.seed(42)

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

print("Environment initialized and libraries imported successfully.")

## 2. The Latent Variable Framework (Intuition)

The foundational intuition for binary models is the Latent Variable Framework.

Assume there is an unobservable, continuous propensity or utility variable Y_star that determines the outcome:
Y_star = X * beta + epsilon

We do not observe Y_star; we only observe the binary realization Y. The observation rule is a threshold mechanism:
Y = 1 if Y_star > 0
Y = 0 if Y_star <= 0

Let's simulate this framework to build intuition.

In [ ]:
# Data Creation: Simulating the Latent Variable Framework
n_samples = 1000
X_raw = np.random.uniform(-4, 4, n_samples)

# Latent utility: Y_star = 1.5 * X + noise
# We use a logistic distribution for the noise to match the Logit model assumption
latent_utility = 1.5 * X_raw + np.random.logistic(loc=0, scale=1, size=n_samples)

# Observe binary outcome based on the threshold (Y_star > 0)
Y_binary = (latent_utility > 0).astype(int)

# Create a DataFrame for easy manipulation
df = pd.DataFrame({
    'Feature_X': X_raw, 
    'Latent_Y_Star': latent_utility, 
    'Observed_Y': Y_binary
})

print("First 5 rows of our synthetic dataset:")
print(df.head())

print("\nSummary Statistics:")
print(df.describe().round(2))

## 3. Visualizing the Latent Framework

Let's visualize how the continuous unobserved variable Y_star translates into the discrete observed variable Y. The zero-line threshold strictly separates the outcomes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: The Latent Variable Y_star
sns.scatterplot(x='Feature_X', y='Latent_Y_Star', hue='Observed_Y', data=df, alpha=0.6, palette='coolwarm', ax=axes[0])
axes[0].axhline(0, color='black', linestyle='--', linewidth=2, label='Threshold (Y_star = 0)')
axes[0].set_title('Unobservable Latent Utility (Y_star) vs Feature X')
axes[0].set_ylabel('Latent Utility Y_star')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: The Observed Binary Variable Y
sns.scatterplot(x='Feature_X', y='Observed_Y', hue='Observed_Y', data=df, alpha=0.3, palette='coolwarm', ax=axes[1])
axes[1].set_title('Observed Binary Outcome (Y) vs Feature X')
axes[1].set_ylabel('Observed Y in {0, 1}')
axes[1].set_yticks([0, 1])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Core Concept 1: The Linear Probability Model (LPM)

The Linear Probability Model applies standard Ordinary Least Squares (OLS) regression directly to a binary outcome. It assumes the link function is the identity function:
P(Y=1|X) = X * beta

Advantage: Coefficients are directly interpretable as marginal effects (the direct change in probability for a 1-unit change in X).
Fatal Flaw: Unbounded predictions. The model will predict probabilities below 0 and above 1 for extreme values of X.

In [ ]:
# Prepare data for statsmodels
X_sm = sm.add_constant(df['Feature_X'])
Y_sm = df['Observed_Y']

# Fit the Linear Probability Model (OLS)
# Note: We MUST use HC3 robust standard errors to correct for inherent heteroskedasticity in LPM
lpm_model = sm.OLS(Y_sm, X_sm).fit(cov_type='HC3')

print("--- Linear Probability Model (LPM) Summary ---")
print(lpm_model.summary().tables[1])

# Check for out-of-bounds predictions
lpm_preds = lpm_model.predict(X_sm)
out_of_bounds = (lpm_preds < 0) | (lpm_preds > 1)
print(f"\nNumber of invalid probability predictions (<0 or >1): {out_of_bounds.sum()} out of {n_samples}")

## 5. Visualizing LPM Flaws: Heteroskedasticity

Aside from predicting impossible probabilities, the LPM severely violates the OLS assumption of homoskedasticity. The variance of a Bernoulli variable is p*(1-p). Since p depends on X, the variance inherently depends on X.

In [ ]:
# Extract residuals from LPM
lpm_residuals = lpm_model.resid

plt.figure(figsize=(8, 5))
plt.scatter(lpm_preds, lpm_residuals, alpha=0.4, color='purple')
plt.axhline(0, color='black', linestyle='-', linewidth=2)
plt.title('LPM Flaw: Heteroskedastic Residuals (Fan Shape)', fontsize=14)
plt.xlabel('LPM Predicted Probability', fontsize=12)
plt.ylabel('Residuals', fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

print("Notice how the spread of residuals changes drastically depending on the predicted probability. This makes standard OLS p-values invalid without robust standard errors.")

## 6. Core Concept 2: The Logit Model

To resolve the boundedness issue, the Logit model assumes the error term epsilon follows a Standard Logistic distribution. The link function is the logit (log-odds) transformation.

P(Y=1|X) = 1 / (1 + exp(-X * beta))

The inverse link function yields the log-odds:
ln(p / (1-p)) = X * beta

We fit this model using Maximum Likelihood Estimation (MLE).

In [ ]:
# Fit the Logit Model via MLE
logit_model = sm.Logit(Y_sm, X_sm).fit(disp=0)

print("--- Logit Model Summary ---")
print(logit_model.summary().tables[1])

print("\nNotice that the Logit coefficient for Feature_X is roughly 1.5, perfectly capturing our true latent generation parameter!")

## 7. Core Concept 3: The Probit Model

The Probit model assumes the error term epsilon follows a Standard Normal distribution. The link function is the inverse standard normal Cumulative Distribution Function (CDF).

While mathematically different (using integrals instead of exponentials), it yields very similar predicted probabilities to the Logit model.

In [ ]:
# Fit the Probit Model via MLE
probit_model = sm.Probit(Y_sm, X_sm).fit(disp=0)

print("--- Probit Model Summary ---")
print(probit_model.summary().tables[1])

# Compare coefficients
results_df = pd.DataFrame({
    'LPM (OLS)': lpm_model.params,
    'Logit (MLE)': logit_model.params,
    'Probit (MLE)': probit_model.params
})

print("\n--- Coefficient Comparison ---")
print(results_df.round(4))
print("\nNote: The scales of the coefficients are different because the underlying latent distributions (Uniform, Logistic, Normal) have different variances.")

## 8. Visualizing the Models: Predicted Probabilities

Let's compare the predicted probabilities of all three models across the feature space to clearly highlight the unbounded failure of the LPM versus the Sigmoid bounds of Logit/Probit.

In [ ]:
# Generate a smooth range of X values for plotting
X_plot_raw = np.linspace(-5, 5, 200)
X_plot_sm = sm.add_constant(X_plot_raw)

# Get probability predictions from all three models
p_lpm = lpm_model.predict(X_plot_sm)
p_logit = logit_model.predict(X_plot_sm)
p_probit = probit_model.predict(X_plot_sm)

plt.figure(figsize=(10, 6))
plt.scatter(df['Feature_X'], df['Observed_Y'], alpha=0.1, color='gray', label='Observed Data')

plt.plot(X_plot_raw, p_lpm, 'r--', label='LPM (Unbounded)', linewidth=2)
plt.plot(X_plot_raw, p_logit, 'b-', label='Logit (Bounded S-Curve)', linewidth=2)
plt.plot(X_plot_raw, p_probit, 'g-.', label='Probit (Bounded S-Curve)', linewidth=2)

# Draw valid probability boundaries
plt.axhline(1, color='black', linestyle=':', alpha=0.5)
plt.axhline(0, color='black', linestyle=':', alpha=0.5)

plt.title('Binary Response Models: Predicted Probabilities Comparison', fontsize=14)
plt.xlabel('Feature X', fontsize=12)
plt.ylabel('Predicted Probability P(Y=1|X)', fontsize=12)
plt.ylim(-0.2, 1.2)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Interpreting Logit Coefficients: Odds Ratios

In a logistic regression model, the estimated coefficients (beta) do NOT represent the change in probability. They represent the change in the LOG-ODDS.

To interpret this multiplicatively, we exponentiate the coefficient to get the Odds Ratio (OR = exp(beta)).
An OR tells us the factor by which the odds of the outcome change for a 1-unit increase in X.

In [ ]:
def extract_odds_ratios(model):
    """Extracts Odds Ratios and 95% Confidence Intervals from a Logit model."""
    params = model.params
    conf_int = model.conf_int()
    
    or_df = pd.DataFrame({
        'Log-Odds (Beta)': params,
        'Odds Ratio (exp(Beta))': np.exp(params),
        'OR 2.5% CI': np.exp(conf_int[0]),
        'OR 97.5% CI': np.exp(conf_int[1])
    })
    return or_df

or_results = extract_odds_ratios(logit_model)
print("--- Odds Ratios and Confidence Intervals ---")
print(or_results.round(3))

feat_x_or = or_results.loc['Feature_X', 'Odds Ratio']
print(f"\nInterpretation: A 1-unit increase in Feature X multiplies the odds of the outcome by {feat_x_or:.2f}.")

## 10. Interpreting Logit Coefficients: Marginal Effects

While Odds Ratios are helpful, businesses often want to know the additive percentage point change in probability. This requires computing the Marginal Effect: dp/dX = beta * p * (1-p).

Because this varies based on p, we compute the Average Marginal Effect (AME) across all observations in the dataset.

In [ ]:
def extract_average_marginal_effects(model, X_data):
    """Computes the Average Marginal Effect (AME) for the Logit model."""
    margeff = model.get_margeff(at='overall', method='dydx')
    
    ame_data = []
    for i, var in enumerate(margeff.margeff):
        ame_data.append({
            'Variable': X_data.columns[i+1], # Skip constant
            'AME (dP/dX)': var,
            'Std Err': margeff.margeff_se[i],
            'P-value': margeff.pvalues[i]
        })
    return pd.DataFrame(ame_data).set_index('Variable')

ame_results = extract_average_marginal_effects(logit_model, X_sm)
print("--- Average Marginal Effects (AME) ---")
print(ame_results.round(4))

feat_x_ame = ame_results.loc['Feature_X', 'AME (dP/dX)']
print(f"\nInterpretation: On average, a 1-unit increase in Feature X increases the probability of the outcome by {(feat_x_ame*100):.1f} percentage points.")

## 11. Visualization Gallery: The Transformation Pipeline

To deeply understand how coefficients work in logistic regression, we can visualize the three spaces: Log-Odds (where the relationship is linear), Odds (where the relationship is exponential), and Probability (where the relationship is S-shaped).

In [ ]:
def visualize_transformation_pipeline():
    x_vals = np.linspace(-4, 4, 200)
    beta_0 = logit_model.params[0]
    beta_1 = logit_model.params[1]
    
    log_odds = beta_0 + beta_1 * x_vals
    odds = np.exp(log_odds)
    probability = 1 / (1 + np.exp(-log_odds))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Plot 1: Log-Odds Space (Linear)
    axes[0].plot(x_vals, log_odds, color='blue', linewidth=2)
    axes[0].set_title('1. Log-Odds Space (Linear)', fontsize=14)
    axes[0].set_xlabel('Predictor X')
    axes[0].set_ylabel('Log-Odds')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Odds Space (Exponential)
    axes[1].plot(x_vals, odds, color='green', linewidth=2)
    axes[1].set_title('2. Odds Space (Exponential)', fontsize=14)
    axes[1].set_xlabel('Predictor X')
    axes[1].set_ylabel('Odds')
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Probability Space (Sigmoid)
    axes[2].plot(x_vals, probability, color='red', linewidth=2)
    axes[2].set_title('3. Probability Space (Non-Linear)', fontsize=14)
    axes[2].set_xlabel('Predictor X')
    axes[2].set_ylabel('Probability P(Y=1|X)')
    axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_transformation_pipeline()

## 12. Practical Example: Credit Risk Modeling

Let's create a realistic multivariate scenario. We are predicting loan defaults (Y=1) based on a continuous variable (Income) and a categorical variable (Missed Payments).

In [ ]:
# Generate Credit Risk Data
n_customers = 3000
income_k = np.random.normal(60, 20, n_customers)
missed_payments = np.random.poisson(0.5, n_customers)

# True data generating process
log_odds_default = -1.0 - 0.05 * income_k + 1.2 * missed_payments
prob_default = 1 / (1 + np.exp(-log_odds_default))
default = np.random.binomial(1, prob_default)

df_credit = pd.DataFrame({
    'Income_K': income_k,
    'Missed_Payments': missed_payments,
    'Default': default
})

print("Synthetic Credit Risk Data Created.")
print(df_credit.head())

## 13. Practice Exercise

Task:
1. Fit a Logistic Regression model to predict 'Default' using 'Income_K' and 'Missed_Payments'.
2. Compute the Odds Ratio for 'Missed_Payments'.
3. Compute the Average Marginal Effect (AME) for 'Income_K'.

In [ ]:
# 1. Fit the model
X_credit = sm.add_constant(df_credit[['Income_K', 'Missed_Payments']])
y_credit = df_credit['Default']
credit_model = sm.Logit(y_credit, X_credit).fit(disp=0)

print("--- Model Summary ---")
print(credit_model.summary().tables[1])

# 2. Odds Ratio for Missed Payments
beta_missed = credit_model.params['Missed_Payments']
or_missed = np.exp(beta_missed)
print(f"\nOdds Ratio for Missed Payments: {or_missed:.2f}")
print(f"Interpretation: Each missed payment multiplies the odds of loan default by approx {or_missed:.2f}.")

# 3. Average Marginal Effect for Income
credit_ame = credit_model.get_margeff(at='overall', method='dydx')
ame_income = credit_ame.margeff[0]
print(f"\nAverage Marginal Effect (AME) for Income_K: {ame_income:.4f}")
print(f"Interpretation: A $1k increase in income decreases the probability of default by {(abs(ame_income)*100):.2f} percentage points, on average.")

## 14. Machine Learning Connections: Logistic Regression

In the machine learning domain, the Logit model is simply known as Logistic Regression. The objective function minimized by ML libraries (like sklearn) is the Binary Cross-Entropy Loss. 

Mathematically, Binary Cross-Entropy is exactly the Negative Log-Likelihood of the Bernoulli distribution. Let's verify that sklearn produces the same coefficients when regularization is disabled.

In [ ]:
# Fit Sklearn Logistic Regression
# penalty=None disables L2 regularization to match standard MLE
sklearn_log_reg = LogisticRegression(penalty=None, fit_intercept=True)
sklearn_log_reg.fit(df_credit[['Income_K', 'Missed_Payments']], df_credit['Default'])

print("--- Framework Comparison ---")
print("Statsmodels (MLE) Coefficients:")
print(f"Income_K:        {credit_model.params['Income_K']:.6f}")
print(f"Missed_Payments: {credit_model.params['Missed_Payments']:.6f}")

print("\nScikit-Learn (Binary Cross-Entropy) Coefficients:")
print(f"Income_K:        {sklearn_log_reg.coef_[0][0]:.6f}")
print(f"Missed_Payments: {sklearn_log_reg.coef_[0][1]:.6f}")

print("\nConclusion: Machine Learning's 'Binary Cross-Entropy' is functionally identical to Statistical 'Maximum Likelihood Estimation'.")

## 15. Edge Case: Complete Separation (Hauck-Donner Effect)

If a predictor perfectly separates the outcomes (e.g., all observations with X > 5 have Y=1), the Maximum Likelihood Estimate for beta diverges to infinity. The optimization algorithm will fail to converge or throw a warning.

In [ ]:
# Simulate Complete Separation
X_sep = np.random.uniform(-5, 5, 200)
# Perfect prediction rule with NO noise
Y_sep = (X_sep > 0).astype(int) 

X_sep_sm = sm.add_constant(X_sep)

try:
    print("Attempting to fit a model with perfectly separated data...")
    # This will trigger a Maximum number of iterations warning or a PerfectSeparationError
    sep_model = sm.Logit(Y_sep, X_sep_sm).fit(disp=0, maxiter=50)
    print("\nSeparation Model Coefficients (Notice they are astronomically high):")
    print(sep_model.params)
except Exception as e:
    print(f"\nModel failed to converge due to error: {e}")

print("\nIn production, complete separation requires Firth's Bias-Reduced Logistic Regression or L1/L2 Regularization to stabilize the coefficients.")

## 16. Summary and Key Takeaways

1. **Linear Probability Model (LPM)**: Unbounded and heteroskedastic. Useful primarily in high-dimensional fixed-effects panel data where non-linear models suffer from the incidental parameters problem.
2. **Logit and Probit**: Bounded probability models. Logit uses the logistic CDF, Probit uses the normal CDF. Logit is strongly preferred in industry due to the interpretability of Odds Ratios.
3. **Log-Odds Space**: Logistic regression coefficients represent linear changes in log-odds, not probabilities.
4. **Odds Ratios**: Computed by exponentiating the coefficient (exp(beta)). They provide a multiplicative framework for understanding how risk scales.
5. **Average Marginal Effects (AME)**: Provide an additive, probability-based interpretation. Because the marginal effect varies across the sigmoid curve, averaging the effect across all actual observations (AME) is the rigorous industry standard.